In [ ]:
!pip install transformers dataset evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import DistilBertTokenizerFast


In [ ]:
print("Booting up the Deep Learning Pipeline...")

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware: {device}")

import pandas as pd
from datasets import Dataset, load_dataset

# Instead of reading from a local CSV, load a common dataset like IMDB for demonstration
print("Downloading IMDB dataset...")
# Load the full IMDB dataset and then sample from it, similar to the original script
imdb_train_dataset = load_dataset("imdb", split="train")
imdb_test_dataset = load_dataset("imdb", split="test")

# Combine train and test splits to sample from the whole pool, then convert to pandas
df_train = imdb_train_dataset.to_pandas()
df_test = imdb_test_dataset.to_pandas()
df_combined = pd.concat([df_train, df_test])

# Sample 4000 entries, similar to the original script
df_sampled = df_combined.sample(4000, random_state=42).reset_index(drop=True)

# Convert the sampled DataFrame back to a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_sampled)

# The IMDB dataset has 'text' and 'label' columns.
# Rename 'label' to 'labels' as expected by Hugging Face models for classification
hf_dataset = hf_dataset.rename_column("label", "labels")

#Loading pre-trained tokenizer
print("Downloading DistilBERT Tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenize Function - now uses the 'text' column from the IMDB dataset
def tokenize_function(examples):
  return tokenizer(examples["text"],padding="max_length",truncation=True,max_length=128)

print("Tokenizing dataset for Neural Network...")
tokenized_datasets=hf_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.train_test_split(test_size=0.2,seed=42)

# Remove the original 'text' column (the '__index_level_0__' column is not present in this dataset, so it's removed)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# Format to PyTorch Tensors
tokenized_datasets.set_format("torch")

print("\n✅ Tokenization Complete! Data is ready for the GPU.")
print(f"Training samples: {len(tokenized_datasets['train'])}")
print(f"Testing samples: {len(tokenized_datasets['test'])}")

Booting up the Deep Learning Pipeline...
Using hardware: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset for Neural Network...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]


✅ Tokenization Complete! Data is ready for the GPU.
Training samples: 3200
Testing samples: 800


In [ ]:
import numpy as np
!pip install evaluate # Ensure evaluate is available
import evaluate
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer

print("Downloading the DistilBERT Neural Network...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

model.to(device)

metrics = evaluate.load("accuracy")

from sklearn.metrics import accuracy_score

# 1. Define our Evaluation Metrics (Bypassing the buggy 'evaluate' library)
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Sometimes models return a tuple of (loss, logits), we just want the logits
    if isinstance(logits, tuple):
        logits = logits[0]

    predictions = np.argmax(logits, axis=-1)

    # Return standard Scikit-Learn accuracy
    return {"accuracy": accuracy_score(labels, predictions)}


print("Setting up hyperparameters...")

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # Fixed variable name: tokenized_dataset -> tokenized_datasets
    eval_dataset=tokenized_datasets["test"],   # Fixed variable name: tokenized_dataset -> tokenized_datasets
    compute_metrics=compute_metrics,
)

print("IGNITION: Starting Neural Network Training...")
print("Grab a coffee, this will take a few minutes on the T4 GPU")
trainer.train()
print("Fine Tuning Completed")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.382958,0.314464,0.881250
2,0.249282,0.340073,0.866250
3,0.154298,0.343901,0.876250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine Tuning Completed


In [ ]:
import shutil
from google.colab import files

print("💾 Saving the model and tokenizer...")
# 1. Save the model weights and the tokenizer rules to a folder
trainer.save_model("./bert_sentiment_model")
tokenizer.save_pretrained("./bert_sentiment_model")

print("📦 Zipping the folder for download...")
# 2. Compress the folder into a zip file
shutil.make_archive("bert_sentiment_model", 'zip', "./bert_sentiment_model")

print("⬇️ Downloading to your computer...")
# 3. Trigger the browser download
files.download("bert_sentiment_model.zip")

💾 Saving the model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

📦 Zipping the folder for download...
⬇️ Downloading to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>